# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Get metadata object
metadata = dataset.metadata
# Print short summary
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets
print("Available Record Sets (@id and name):")
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    for rs in metadata.record_sets:
        print(f"@id: {rs['@id']} | Name: {rs.get('name', '')}")
else:
    # Fallback approach: load the raw Croissant JSON-LD to extract recordSet IDs
    import requests
    schema_json = requests.get(croissant_url).json()
    record_sets = []
    def find_record_sets(obj):
        if isinstance(obj, dict):
            if obj.get('@type') == 'cr:RecordSet':
                record_sets.append(obj)
            for v in obj.values():
                find_record_sets(v)
        elif isinstance(obj, list):
            for i in obj:
                find_record_sets(i)
    find_record_sets(schema_json)
    for rs in record_sets:
        print(f"@id: {rs['@id']}")

## For this dataset, let's explore fields (columns) for the primary record set. Let's print out field information:
if record_sets:
    main_recordset = record_sets[0]['@id']
else:
    main_recordset = None

def print_fields(recordset_id, schema_json):
    # locate the recordset object
    for obj in record_sets:
        if obj['@id'] == recordset_id:
            recordset_obj = obj
            fields = recordset_obj.get('field', [])
            if isinstance(fields, dict):
                fields = [fields]
            print(f"\nFields for RecordSet {recordset_id}:")
            for field_obj in fields:
                if isinstance(field_obj, dict):
                    print(f"  Field @id: {field_obj.get('@id')}, name: {field_obj.get('name','')}, dataType: {field_obj.get('dataType','')}")
                elif isinstance(field_obj, str):
                    print(f"  Field ref: {field_obj}")
            break

if main_recordset:
    print_fields(main_recordset, schema_json)
else:
    print("No record sets found in the schema.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Use the discovered record set IDs for extraction
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    # Try loading records; may raise if not supported
    try:
        records_gen = dataset.records(record_set=record_set_id)
        records = list(records_gen)
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f" - Loaded {len(records)} records. Columns: {dataframes[record_set_id].columns.tolist()}")
        else:
            print(f" - No records found for {record_set_id}.")
    except Exception as e:
        print(f" - Error loading records for {record_set_id}: {e}")

# Choose the main record set to work with:
active_record_set_id = None
for rsid in record_set_ids:
    if rsid in dataframes and not dataframes[rsid].empty:
        active_record_set_id = rsid
        break

if active_record_set_id:
    print(f"\nActive Record Set: {active_record_set_id}")
    print("Columns:", dataframes[active_record_set_id].columns.tolist())
    display(dataframes[active_record_set_id].head())
else:
    print("No records available to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
if active_record_set_id:
    df = dataframes[active_record_set_id].copy()
    # Attempt to automatically select a numeric field for demonstration
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # If not found, try to convert some known fields that may be numeric
    if not numeric_field_id:
        for col in df.columns:
            # Heuristic: 'Coverage', 'Peptides', 'MW', or 'pI' are likely numeric in this domain
            if any(keyword in col.lower() for keyword in ['coverage', 'peptide', 'mw', 'pi', 'abundance', 'count', 'score']):
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                    if df[col].notna().sum() > 0:
                        numeric_field_id = col
                        break
                except Exception:
                    continue

    if numeric_field_id is None:
        print("No obvious numeric field found for EDA.")
    else:
        print(f"Using field '{numeric_field_id}' for numeric analysis.")
        # Filter: values > threshold
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean value): {len(filtered_df)} records")
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' field (first 5 rows):")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a categorical field
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() < 20 and df[col].dtype == object:
                group_field = col
                break
        if group_field:
            print(f"Grouping normalized data by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if active_record_set_id and numeric_field_id is not None:
    # Plot distribution of numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()

    # If there is a grouping field, plot group means
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,4))
        group_means = df.groupby(group_field)[numeric_field_id].mean().dropna()
        group_means.plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we initialized the `mlcroissant` library and loaded the metadata and record sets from the Croissant dataset schema.
* The dataset includes mass spectrometry protein analysis of extracellular vesicles isolated from human mast cells, with fields such as accession, peptide counts, molecular weight, etc.
* We programmatically reviewed available record set and field `@id`s to ensure robust referencing and loaded the data to pandas DataFrames.
* Exploratory analysis included filtering and normalization of a selected numeric field, with optional grouping by a categorical attribute, and data visualization of distributions and category means.

Further analyses could include advanced visualizations, statistical testing, or deeper domain-specific inspection of modifications and peptide sequences as defined by the dataset content.